# Gordian v2 — Colab screening notebook

For **light-compute, exploratory** Phase-4-style work only (candidate mechanism screens, feasibility checks, quick diagnostics). NOT for anything that touches the holdout, writes to `research/decisions.md`, or runs `validation.run_battery` / `log_trial`. Those run in the full project environment, where the quarantine guard and the review trail apply.

**Convention match:** this notebook reuses the project's frozen source (`src/signals.py`, `src/panel_factory.py`, `src/decomposition.py`, `validation/ic.py`, `validation/canaries.py`) and the frozen train/val panel (`data/panel/trainval_*.csv`, through 2021-12-31 only, **never the holdout**).

**Workflow:** run a screen here, get numbers, then take the exported result back to the project environment, where it is audited and logged to `research/decisions.md` before it counts as a finding. This notebook logs nothing; it is scratch compute, with the same evidentiary status as an exploratory REPL run.

**Non-trial-counted by construction** (no `run_battery` call is made anywhere below): exploratory candidate screens do not consume the 250-trial budget.


## 1. Setup — clone repo + frozen data

The repo carries the signal/validation source AND the committed frozen train/val panel CSVs, so a single `git clone` gives a screen everything it needs.


In [ ]:
# --- Clone the repo (source + frozen train/val panel) ---
!git clone https://github.com/Aymo-WH/trading-bot.git /content/repo
%cd /content/repo/v2
!git log --oneline -1


In [ ]:
# --- No separate data pull needed ---
# The clone above already contains the frozen train/val panel
# (data/panel/trainval_*.csv, eligibility_trainval.csv, MANIFEST.json) plus all
# the src/ + validation/ source a screen imports. Nothing else to fetch.
print("using repo clone — no extra data pull required")


In [ ]:
# --- Sanity check: confirm the frozen panel is present and matches the
# expected shape/date range before trusting anything downstream. ---
import json

with open("data/panel/MANIFEST.json") as f:
    manifest = json.load(f)

print("panel_start:", manifest["panel_start"])
print("trainval_end:", manifest["trainval_end"], "(frozen quarantine boundary — never cross this)")
print("n_tickers:", manifest["n_tickers"])
print("universe_rule:", manifest["universe_rule"])

## 2. Load the frozen train/val panel

Same loader as `src/portfolio_m0.load_canonical_trainval()` — reused directly rather than reimplemented, so this notebook can't silently drift from the frozen construction.

In [ ]:
import sys
sys.path.insert(0, "/content/repo/v2")

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from src.portfolio_m0 import load_canonical_trainval
from src.signals import (
    momentum_s1, trend_s2, low_beta_s3, seasonality_s4,
    daily_returns, equal_weight_market_return, _cross_sectional_zscore,
)
from src.panel_factory import rebalance_dates
from src.decomposition import MIN_PRIOR_OBS, expanding_static_timing
from validation.ic import cross_sectional_ic, ic_summary, passes_graduation
from validation.canaries import run_signal_canaries

close, elig, categories, manifest = load_canonical_trainval()
print("panel shape:", close.shape, "| date range:", close.index.min().date(), "to", close.index.max().date())
print("categories:\n", categories.value_counts())

## 3. Reference: existing Tier-1 signals + baseline context

So a new candidate can be checked against what has already been tried, without re-deriving it. Rows marked *later screen* come from Phase-4 exploratory screens run after the decisions log in this repo ends (D40); their scripts are not included here.

| Signal | Status | Note |
|---|---|---|
| S1 momentum | Graduated | t_nw≈3.92 |
| S2 TS trend | Graduated | t_nw≈3.16, ρ(S1,S2)≈0.796 |
| S3 low-beta (BAB) | Failed, wrong-signed | t=-3.06 — defensive axis has NEGATIVE IC on this sample |
| S4 seasonality | Failed, null | t=-0.22 |
| S5 carry (static tilt) | Graduated but flagged | NOT timing — a constant full-sample ordering reproduces 98% of raw IC; static risk-premium tilt, fails time_shift canary |
| Idio-vol/dispersion (later screen) | Closed, no promotion | Collapses to null once S3 is regressed out — same defensive axis as S3, not distinct |
| Network-momentum (later screen) | Closed, null | — |
| Bond-sleeve Amihud illiquidity (later screen) | Closed | ρ=0.867 with the same safe→risky credit axis — not distinct |

**The core finding that matters for any new candidate:** S1, S2 and S5_carry all converge on the SAME underlying static risk-premium tilt (later screens measured S1_static/S2_static at ρ=0.9985). Effective breadth is ~1.11 (D25), not 2-3 signals. **A new candidate is only useful if it has a genuinely different economic MECHANISM**, not just low historical correlation: something can look uncorrelated on paper and still share the same "risky beats defensive" exposure, and collapse the same way idio-vol, network-momentum and bond-liquidity all did.

**Still open, not yet tested:** individual-CUSIP repo specialness / collateral scarcity, a dealer-balance-sheet-driven mechanism not correlated with credit quality. Two open questions: (1) does single-CUSIP specialness survive averaging into a basket-level ETF (a literature question); (2) it needs a new data dependency (repo specialness data is not in FRED/yfinance), which needs a logged decision before any data is pulled, the same pattern as D28/D32/D37.


## 4. Screen template

Build the candidate signal, get a raw IC read, then run it through the full canary battery before trusting anything. Edit the `build_candidate_signal` function for whatever candidate you are testing; everything below it is reusable.


In [ ]:
VOL_WINDOW = 252  # matches src.signals.BETA_WINDOW, reuse by convention

def build_candidate_signal(close: pd.DataFrame, elig: pd.DataFrame) -> pd.DataFrame:
    """EDIT THIS — construct your candidate signal here.

    Must return a DataFrame indexed by date, one column per asset, PIT
    (value at date t uses only data at/before t), NaN where undefined or
    ineligible. Higher score = more favored, by convention (long high,
    short low), matching src/signals.py.
    """
    raise NotImplementedError("fill in the candidate construction")


def run_screen(signal_fn, close, elig, label: str, fwd_horizon_days: int = 5):
    """Runs the standard exploratory-screen pipeline: build -> rebalance-date
    align -> raw IC -> canary battery. Same pipeline as the Phase-4 screens."""
    rebal = rebalance_dates(close.index)
    window = rebal[(rebal >= pd.Timestamp(manifest["panel_start"]))
                    & (rebal <= pd.Timestamp(manifest["trainval_end"]))]

    sig_raw = signal_fn(close, elig)
    elig_reb = elig.reindex(window).astype(bool)
    sig_z = _cross_sectional_zscore(sig_raw.reindex(window), elig_reb)

    fwd = close.shift(-fwd_horizon_days) / close - 1.0

    ic = cross_sectional_ic(sig_z, fwd, min_names=10)
    summary = ic_summary(ic, nw_lags=2)
    print(f"[{label}] mean_ic={summary['mean_ic']:.4f} t_nw={summary['t_nw']:.3f} "
          f"pct_years_positive={summary['pct_years_positive']:.2f} n_dates={summary['n_dates']}")
    print(f"[{label}] passes_graduation={passes_graduation(summary)}")

    print(f"[{label}] running canary battery (label_shuffle does 100 reruns "
          f"— this is the slow part, expect a few minutes)...")
    canaries = run_signal_canaries(sig_z, fwd)
    print(f"[{label}] canaries all_passed={canaries['all_passed']}")
    for r in canaries["results"]:
        print(f"  {r['canary']}: passed={r['passed']}")

    return {"label": label, "signal": sig_z, "ic_summary": summary, "canaries": canaries}


# Example:
# result = run_screen(build_candidate_signal, close, elig, label="my_candidate")

## 5. Mechanism-distinctness check against existing signals

The decisive test used on every candidate so far (idio-vol vs S3, network-momentum, bond-liquidity vs S1/S2/S5): does the candidate's edge SURVIVE regressing out the composite + existing statics, or does it collapse to null? Collapse = "same mechanism, different construction", not a new candidate.

In [ ]:
def cross_sectional_regress_out(y: pd.DataFrame, xs: list, elig: pd.DataFrame) -> pd.DataFrame:
    """Per-date OLS residual of y against the given regressors (all PIT).
    Same pattern as the Phase-4 screens."""
    resid = pd.DataFrame(index=y.index, columns=y.columns, dtype=float)
    min_obs = len(xs) + 1 + 5
    for dt in y.index:
        e = elig.loc[dt] if dt in elig.index else pd.Series(False, index=y.columns)
        cols = [y.loc[dt]] + [x.loc[dt] if dt in x.index else pd.Series(np.nan, index=y.columns) for x in xs]
        mask = e.reindex(y.columns).fillna(False)
        for c in cols:
            mask = mask & c.reindex(y.columns).notna()
        if mask.sum() < min_obs:
            continue
        names = y.columns[mask]
        Y = y.loc[dt, names].to_numpy(dtype=float)
        X = np.column_stack([np.ones(len(names))] + [x.loc[dt, names].to_numpy(dtype=float) for x in xs])
        beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
        resid.loc[dt, names] = Y - X @ beta
    return resid


def build_existing_regressors(close, elig, window):
    """The standard control set: S1+S2 composite, plus S1/S2 static tilts
    (these carry the shared risk-premium axis)."""
    s1_raw = momentum_s1(close, elig)
    s2_raw = trend_s2(close, elig)
    composite_w = (0.5 * (s1_raw + s2_raw)).reindex(window)
    s1_static, _ = expanding_static_timing(s1_raw.reindex(window), min_prior_obs=MIN_PRIOR_OBS)
    s2_static, _ = expanding_static_timing(s2_raw.reindex(window), min_prior_obs=MIN_PRIOR_OBS)
    return [composite_w, s1_static, s2_static]


# Example usage once you have a candidate signal built:
# rebal = rebalance_dates(close.index)
# window = rebal[(rebal >= pd.Timestamp(manifest["panel_start"])) & (rebal <= pd.Timestamp(manifest["trainval_end"]))]
# elig_reb = elig.reindex(window).astype(bool)
# regressors = build_existing_regressors(close, elig, window)
# candidate_z = _cross_sectional_zscore(build_candidate_signal(close, elig).reindex(window), elig_reb)
# residual = cross_sectional_regress_out(candidate_z, regressors, elig_reb)
# fwd5 = close.shift(-5) / close - 1.0
# resid_summary = ic_summary(cross_sectional_ic(residual, fwd5, min_names=10), nw_lags=2)
# print("residual (mechanism-distinct test) t_nw:", resid_summary["t_nw"])
# print("-> if this collapses toward 0 / loses significance, it's the same mechanism, not a new one")

## 6. Export result for review

This notebook cannot write to `research/decisions.md` itself. Export a result JSON in the same shape the project's screens use, then take it back to the project environment to be audited and logged before it is trusted.


In [ ]:
import json

def export_result(result: dict, path: str = "/content/candidate_result.json"):
    out = {
        "label": result["label"],
        "ic_summary": result["ic_summary"],
        "canaries": result["canaries"],
        "note": ("UNAUDITED — generated in Colab, not leak-audited, "
                 "not logged to decisions.md. Audit it in the project "
                 "environment before treating it as a project finding."),
    }
    with open(path, "w") as f:
        json.dump(out, f, indent=2, default=str)
    print(f"wrote {path} — download it (Colab: folder icon on the left, right-click > Download) "
          f"and audit it in the project environment.")

# export_result(result)
